# Deploy NVIDIA NIM from AWS Marketplace

NVIDIA NIM, a component of NVIDIA AI Enterprise, enhances your applications with the power of state-of-the-art large language models (LLMs), providing unmatched natural language processing and understanding capabilities. Whether you're developing chatbots, content analyzers, or any application that needs to understand and generate human language, NVIDIA NIM for LLMs has you covered.

In this example we show how to deploy the **NVIDIA Nemotron-3-Super-120B-A12B** model from AWS Marketplace on Amazon SageMaker.

Nemotron-3-Super-120B-A12B is a large language model (LLM) trained by NVIDIA, designed to deliver strong agentic, reasoning, and conversational capabilities. It is optimized for collaborative agents and high-volume workloads such as IT ticket automation. The model employs a hybrid **Latent Mixture-of-Experts (LatentMoE)** architecture with **Multi-Token Prediction (MTP)** layers for faster text generation. It has **12B active parameters** and **120B total parameters**.

The model's reasoning capabilities can be configured through a flag in the chat template, supporting reasoning on, reasoning off, and low-effort reasoning modes.

**Supported languages:** English, French, German, Italian, Japanese, Spanish, and Chinese.

Please check out the [NIM LLM docs](https://docs.nvidia.com/nim/large-language-models/latest/introduction.html) for more information.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. To deploy this ML model successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**  
    2. or your AWS account has a subscription to one of the models listed above.

**⚠️ Disclaimer**

Reasoning models often require longer inference times, which may exceed the default 60-second timeout limit for [**AWS SageMaker's non-streaming endpoints**](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_runtime_InvokeEndpoint.html). This notebook shows examples for both the streaming and non-streaming endpoints.

To avoid inference failures due to timeout:
- It is **recommended** to use the **SageMaker streaming endpoint** for this model as shown in the [Streaming inference](#Streaming-inference) section of this notebook.
- If your use case **requires** using a **non-streaming endpoint** as shown in the [Run Inference](#Run-Inference) section of this notebook, you must first contact **AWS Support** to request an increased timeout limit for your **AWS Account and Region** to avoid unexpected errors.

## Subscribe to the model package
To subscribe to the model package:
1. Open the model package listing page
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
import boto3, json, sagemaker, time, os
from sagemaker import get_execution_role, ModelPackage
from botocore.config import Config

config = Config(read_timeout=3600)
sess = boto3.Session()
sm = sess.client("sagemaker")
sagemaker_session = sagemaker.Session(boto_session=sess)
role = get_execution_role()
client = boto3.client("sagemaker-runtime", config=config)
region = sess.region_name

In [ ]:
# replace the arn below with the model package arn you want to deploy
nim_package = "ENTER PACKAGE NAME"

# Mapping for Model Packages
model_package_map = {
    "us-east-1": f"arn:aws:sagemaker:us-east-1:865070037744:model-package/{nim_package}",
    "us-east-2": f"arn:aws:sagemaker:us-east-2:057799348421:model-package/{nim_package}",
    "us-west-1": f"arn:aws:sagemaker:us-west-1:382657785993:model-package/{nim_package}",
    "us-west-2": f"arn:aws:sagemaker:us-west-2:594846645681:model-package/{nim_package}",
    "ca-central-1": f"arn:aws:sagemaker:ca-central-1:470592106596:model-package/{nim_package}",
    "eu-central-1": f"arn:aws:sagemaker:eu-central-1:446921602837:model-package/{nim_package}",
    "eu-west-1": f"arn:aws:sagemaker:eu-west-1:985815980388:model-package/{nim_package}",
    "eu-west-2": f"arn:aws:sagemaker:eu-west-2:856760150666:model-package/{nim_package}",
    "eu-west-3": f"arn:aws:sagemaker:eu-west-3:843114510376:model-package/{nim_package}",
    "eu-north-1": f"arn:aws:sagemaker:eu-north-1:136758871317:model-package/{nim_package}",
    "ap-southeast-1": f"arn:aws:sagemaker:ap-southeast-1:192199979996:model-package/{nim_package}",
    "ap-southeast-2": f"arn:aws:sagemaker:ap-southeast-2:666831318237:model-package/{nim_package}",
    "ap-northeast-2": f"arn:aws:sagemaker:ap-northeast-2:745090734665:model-package/{nim_package}",
    "ap-northeast-1": f"arn:aws:sagemaker:ap-northeast-1:977537786026:model-package/{nim_package}",
    "ap-south-1": f"arn:aws:sagemaker:ap-south-1:077584701553:model-package/{nim_package}",
    "sa-east-1": f"arn:aws:sagemaker:sa-east-1:270155090741:model-package/{nim_package}",
}

region = boto3.Session().region_name
if region not in model_package_map.keys():
    raise Exception(f"Current boto3 session region {region} is not supported.")

model_package_arn = model_package_map[region]
model_package_arn

## Create the SageMaker Endpoint

We first define SageMaker model using the specified ModelPackageArn.

In [ ]:
# Define the model details
sm_model_name = "nim-nemotron-3-super-120b-a12b"

# Create the SageMaker model
create_model_response = sm.create_model(
    ModelName=sm_model_name,
    PrimaryContainer={
        'ModelPackageName': model_package_arn
    },
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True
)
print("Model Arn: " + create_model_response["ModelArn"])

Next we create endpoint configuration specifying instance type.

In [ ]:
# Create the endpoint configuration
endpoint_config_name = sm_model_name

create_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': sm_model_name,
            'InitialInstanceCount': 1,
            'InstanceType': 'ml.p5.48xlarge',
            'InferenceAmiVersion': 'al2-ami-sagemaker-inference-gpu-2',
            'RoutingConfig': {'RoutingStrategy': 'LEAST_OUTSTANDING_REQUESTS'},
            'ModelDataDownloadTimeoutInSeconds': 3600,
            'ContainerStartupHealthCheckTimeoutInSeconds': 3600,
        }
    ]
)
print("Endpoint Config Arn: " + create_endpoint_config_response["EndpointConfigArn"])

Using the above endpoint configuration we create a new sagemaker endpoint and wait for the deployment to finish. The status will change to InService once the deployment is successful.

In [ ]:
# Create the endpoint
endpoint_name = endpoint_config_name
create_endpoint_response = sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name
)

print("Endpoint Arn: " + create_endpoint_response["EndpointArn"])

In [ ]:
resp = sm.describe_endpoint(EndpointName=endpoint_name)
status = resp["EndpointStatus"]
print("Status: " + status)

while status == "Creating":
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=endpoint_name)
    status = resp["EndpointStatus"]
    print("Status: " + status)

print("Arn: " + resp["EndpointArn"])
print("Status: " + status)

### Run Inference

Once we have the model deployed we can use a sample text to do an inference request. NIM on SageMaker supports the OpenAI API inference protocol. For explanation of supported parameters please see [this link](https://docs.api.nvidia.com/nim/reference/nvidia-llama-3_3-nemotron-super-49b-v1-infer).

Nemotron-3-Super-120B-A12B supports configurable reasoning via `chat_template_kwargs`:
- **Reasoning On**: `{"enable_thinking": True}` — model generates internal reasoning trace before responding
- **Reasoning Off**: `{"enable_thinking": False}` — model responds directly without reasoning trace
- **Low Effort Reasoning**: `{"enable_thinking": True, "low_effort": True}` — shorter reasoning trace for simple queries

In [ ]:
payload_model = "nvidia/nemotron-3-super-120b-a12b"

#### Non-streaming (Reasoning Off)

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is NVIDIA? Answer in 2-3 sentences."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 512,
    "stream": False,
    "chat_template_kwargs": {"enable_thinking": False}
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload)
)

output = json.loads(response["Body"].read().decode("utf8"))
print(json.dumps(output, indent=2))

#### Non-streaming (Reasoning On)

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is NVIDIA? Answer in 2-3 sentences."
        }
    ],
    "temperature": 0.2,
    "max_tokens": 3000,
    "stream": False,
    "chat_template_kwargs": {"enable_thinking": True}
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload)
)

output = json.loads(response["Body"].read().decode("utf8"))
print(json.dumps(output, indent=2))

#### Non-streaming (Low Effort Reasoning)

Low effort reasoning uses significantly fewer tokens than standard reasoning, recommended for simple queries.

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is the capital of Japan?"
        }
    ],
    "temperature": 1.0,
    "top_p": 0.95,
    "max_tokens": 1024,
    "stream": False,
    "chat_template_kwargs": {
        "enable_thinking": True,
        "low_effort": True
    }
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps(payload)
)

output = json.loads(response["Body"].read().decode("utf8"))
print(json.dumps(output, indent=2))

### Streaming inference

NIM on SageMaker also supports streaming inference and you can enable that by setting **`"stream"` as `True`** in the payload and by using [`invoke_endpoint_with_response_stream`](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker-runtime/client/invoke_endpoint_with_response_stream.html) method.

#### Streaming (Reasoning Off)

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is NVIDIA? Answer in 2-3 sentences."
        }
    ],
    "max_tokens": 512,
    "stream": True,
    "extra_body": {
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
}

response = client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    Body=json.dumps(payload),
    ContentType="application/json",
    Accept="application/jsonlines",
)

In [ ]:
buf = b""
stream_finished = False

for event in response["Body"]:
    b = event.get("PayloadPart", {}).get("Bytes", b"")
    if not b:
        continue
    buf += b

    text = buf.decode("utf-8-sig", errors="ignore")
    lines, _, tail = text.rpartition("\n")
    buf = tail.encode("utf-8", errors="ignore")

    for line in lines.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line == "data: [DONE]":
            print("\n[DONE]")
            stream_finished = True
            break
        if line.startswith("data:"):
            line = line[len("data:"):].strip()

        data = json.loads(line)
        content = (data.get("choices") or [{}])[0].get("delta", {}).get("content", "")
        if content:
            print(content, end="", flush=True)

#### Streaming (Reasoning On)

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is NVIDIA? Answer in 2-3 sentences."
        }
    ],
    "max_tokens": 3000,
    "stream": True,
    "extra_body": {
        "chat_template_kwargs": {
            "enable_thinking": True
        }
    }
}

response = client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    Body=json.dumps(payload),
    ContentType="application/json",
    Accept="application/jsonlines",
)

In [ ]:
buf = b""
stream_finished = False

for event in response["Body"]:
    b = event.get("PayloadPart", {}).get("Bytes", b"")
    if not b:
        continue
    buf += b

    text = buf.decode("utf-8-sig", errors="ignore")
    lines, _, tail = text.rpartition("\n")
    buf = tail.encode("utf-8", errors="ignore")

    for line in lines.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line == "data: [DONE]":
            print("\n[DONE]")
            stream_finished = True
            break
        if line.startswith("data:"):
            line = line[len("data:"):].strip()

        data = json.loads(line)
        content = (data.get("choices") or [{}])[0].get("delta", {}).get("content", "")
        if content:
            print(content, end="", flush=True)

#### Streaming (Low Effort Reasoning)

In [ ]:
payload = {
    "model": payload_model,
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "What is the capital of Japan?"
        }
    ],
    "temperature": 1.0,
    "top_p": 0.95,
    "max_tokens": 1024,
    "stream": True,
    "extra_body": {
        "chat_template_kwargs": {
            "enable_thinking": True,
            "low_effort": True
        }
    }
}

response = client.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    Body=json.dumps(payload),
    ContentType="application/json",
    Accept="application/jsonlines",
)

In [ ]:
buf = b""
stream_finished = False

for event in response["Body"]:
    b = event.get("PayloadPart", {}).get("Bytes", b"")
    if not b:
        continue
    buf += b

    text = buf.decode("utf-8-sig", errors="ignore")
    lines, _, tail = text.rpartition("\n")
    buf = tail.encode("utf-8", errors="ignore")

    for line in lines.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line == "data: [DONE]":
            print("\n[DONE]")
            stream_finished = True
            break
        if line.startswith("data:"):
            line = line[len("data:"):].strip()

        data = json.loads(line)
        content = (data.get("choices") or [{}])[0].get("delta", {}).get("content", "")
        if content:
            print(content, end="", flush=True)

### Terminate endpoint and clean up artifacts

In [ ]:
sm.delete_model(ModelName=sm_model_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_endpoint(EndpointName=endpoint_name)